# Data Workflow: Database and Market Data Integration

## Overview

This notebook explains how simulated fund, position, investor and market data are loaded, enriched and prepared for risk workflows. It focuses on the data path from source files and reference data to SQLite tables and risk-ready inputs used by the fund notebooks.

The objective is to make the data workflow transparent before reviewing the risk calculations.

This is not a risk report. It shows how the repository builds a controlled fund dataset, stores it in SQLite, enriches the raw position data, and makes the resulting data available to the risk analytics modules.

The project uses controlled mock data with a fixed reference date of `2026-05-13` so the examples are reproducible.

In [ ]:
from src.config import VALUATION_DATE
VALUATION_DATE




## Data workflow

The workflow has four layers:

1. **Source data layer**

   * controlled JSON inputs under `reference_data/`
   * generated position files under `data/`

2. **SQLite database layer**

   * SQLite database created by `setup_db.py`
   * fund metadata and raw position data loaded into the database

3. **Market data and enrichment layer**

   * Bloomberg-style market data interface
   * `bdp()` for point-in-time fields
   * `bdh()` for historical series
   * enrichment of raw positions with market-data and fund-admin fields

4. **Analytics input layer**

   * risk-ready DataFrame loaded from the database
   * input for VaR, stress, liquidity, leverage, and reporting workflows


### 1. Source data layer

The repository uses controlled inputs under `reference_data/` as the starting point for the example dataset.

Fund-level reference data is stored under `reference_data/funds/`:

* `fund_registry.json` defines which funds are loaded into the database.
* each `fund_profile.json` stores static fund metadata used to build the `funds` table.
* each `risk_policy.json` stores fund-specific internal monitoring choices, including modelling assumptions and thresholds where these are defined by the fund's risk framework rather than by regulation.

Portfolio specifications define the holdings used to generate reproducible input files under `data/`.

For position-based funds such as the hedge fund and UCITS, holdings are defined through position specification files and loaded into the SQLite `positions` table. 



In [ ]:
%cd ../..
!ls -1 reference_data/funds/*/position_specs.json


## 2. SQLite database layer

The SQLite database is created by running `setup_db.py`.

This script creates the database structure, loads fund metadata from the fund-level reference JSON files, and populates the raw `positions` table from the generated position files.

The database is created locally at: `<project_root>/data/risk_management.db`

Create the database once when setting up the project on a new machine. Recreate it only when the input data or database structure changes.


In [ ]:
# Setup (do this once or when rebuilding the database)
from src.setup_db import run as setup_db

# This creates/rebuilds the database from reference_data/ and Excel position files
setup_db(force=True)

print(f'Valuation date: {VALUATION_DATE}')
print('Database ready at: data/risk_management.db')


### Database schema

The SQLite database contains tables for fund metadata and position-based models.

| Group                | Tables                                                                                                                                                                 | Purpose                                                                                         |
| -------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------- |
| Fund metadata        | `funds`                                                                                                                                                                | Metadata for all funds loaded from `fund_registry.json` and each fund's `fund_profile.json` |
| Position-based model | `positions`, `positions_enriched`, `instruments`                                                                                                                       | Raw and enriched holdings used by the risk workflows                             |


### Connecting to the database

Once the database has been created, it can be accessed through the repository's database utilities in `src.data.database`.

The repository provides a helper function, `get_engine()`, which returns a configured SQLAlchemy engine. This centralises database configuration and avoids repeating connection setup throughout the notebooks.

In [ ]:
from src.data.database import get_engine

engine = get_engine()
type(engine)

### Querying the database

Once connected, the database can be queried directly using SQLAlchemy and standard SQL. See the following examples.

In [ ]:
from sqlalchemy import text
import pandas as pd

pd.read_sql(
    text("""
    SELECT fund_id
    FROM funds
    ORDER BY fund_id
    LIMIT 10
    """),
    engine,
)

In [ ]:
for table in ["funds", "positions"]:
    print(f"\n{table}")
    display(pd.read_sql(text(f"SELECT * FROM {table} LIMIT 5"), engine))


In [ ]:
import pandas as pd
pd.options.display.float_format = "{:,.2f}".format

from sqlalchemy import text

pd.read_sql(
    text("""
    SELECT *
    FROM funds
    ORDER BY fund_id
    LIMIT 10
    """),
    engine,
)

In [ ]:
from src.data.database import get_engine
import pandas as pd
from sqlalchemy import text

engine = get_engine()

# All tables
tables_df = pd.read_sql(
    text("SELECT name FROM sqlite_master WHERE type='table'"),
    engine
)
print("Tables:", tables_df['name'].tolist())

# Columns in positions table
cols_df = pd.read_sql(
    text("PRAGMA table_info(positions)"),
    engine
)
print("\nPositions columns:")
print(cols_df[['name', 'type']])


In [ ]:
fund_id = "AIFM_HedgeFund"  # ← Full fund ID, not 'AIFM_HF'
valuation_date = "2026-05-13"

query = text("""
SELECT *
FROM positions
WHERE fund_id = :fund_id
  AND position_date = :position_date
""")

positions = pd.read_sql(
    query,
    engine,
    params={
        "fund_id": fund_id,
        "position_date": valuation_date,  
    },
)

positions.head(2)


Although the repository remains fully accessible through SQL, most notebooks use reusable query functions provided by the repository rather than embedding SQL directly in notebook cells. This keeps the notebooks focused on risk workflows and reporting while keeping common data access logic in one place.

Examples include helper functions for retrieving positions, NAV history, enriched risk-ready datasets, and other frequently used data structures, as shown later in this notebook.


**Ex. 1: Querying fund positions**

The primary query function for daily position snapshots:

In [ ]:
from src.data.database import query_positions
import pandas as pd

# Get all positions for one fund on one date
fund_id = 'AIFM_HedgeFund'
valuation_date = '2026-05-13'

positions = query_positions(engine, fund_id, valuation_date)
positions.head(2)

**Ex. 2: Querying historical NAV and P&L**

In [ ]:
from src.data.database import query_nav_history
# Load 250-day NAV history
nav_hist = query_nav_history(engine, 'AIFM_HedgeFund')
nav_hist['date'] = pd.to_datetime(nav_hist['date'])

print(f'NAV history: {len(nav_hist)} days')
nav_hist.tail()

## 3. Market data and enrichment layer

The repository includes a lightweight market data interface that mirrors the type of workflow normally handled through Bloomberg or another market data vendor.

The interface exposes Bloomberg-style methods:

- `bdp()` for point-in-time fields
- `bdh()` for historical series

It also uses Bloomberg-style field names, such as:

- `PX_LAST`
- `BETA`
- `DUR_ADJ_MID`
- `CONVEXITY`
- `YLD_YTM_MID`

In this project, the interface is backed by controlled mock data, `yfinance` where appropriate, and local caching for downloaded time series. This keeps the examples reproducible while keeping the analytics code independent from the underlying data source.

The same workflow can therefore use the mock interface in this repository and a real vendor adapter in a production setting.



In [ ]:
from src.data.mock_bloomberg import MockBloomberg

# Create a mock Bloomberg instance
bbg = MockBloomberg(seed=42)  # seed for reproducibility

# from the object below we can call methods like bbg.get_price('AAPL', '2026-05-13') to get mock price data
type(bbg)

### bdp: Bloomberg Data Point (static fields)

Get current or latest field values for a security:

In [ ]:
# equity example
fields = ['BETA', 'NAME', 'CRNCY', 'VOLUME_AVG_20D']
result_eq = bbg.bdp('AAPL US Equity', fields)
print('AAPL US Equity:')
result_eq

In [ ]:
# bond example
fields_bond = ['DUR_ADJ_MID', 'CONVEXITY', 'YLD_YTM_MID']
result_bond = bbg.bdp('US912828YK09 Govt', fields_bond)  
result_bond



### bdh: Bloomberg Data History (time series)

Get historical prices for backtesting:

In [ ]:
# Get 60 days of price history
hist = bbg.bdh('AAPL US Equity', 'PX_LAST', '2026-03-01', '2026-05-13')
print(f'AAPL price history: {len(hist)} days')
print('...')
hist.tail(5)

<small>
Note: Historical market data downloaded through the mock market data interface may be cached locally under <code>data/yf_cache/</code>. The cache is only a development convenience and can be refreshed when the requested date range is not covered.
</small>

### Enrichment

Raw positions from the database include quantity and price, but do not always include risk sensitivities such as beta, duration, or convexity. Enrichment adds these fields from market data sources.

**For liquid assets** such as equities, bonds, FX instruments, and listed securities with market identifiers:

* call the mock Bloomberg-style interface to fetch sensitivities such as `BETA`, `DUR_ADJ_MID`, `CONVEXITY`, and `YLD_YTM_MID`
* store the enriched records in the `positions_enriched` table

| Asset Class     | Columns                                     | Source                                       |
| --------------- | ------------------------------------------- | -------------------------------------------- |
| Equity          | beta, eqy_dvd_yld_ind, volume_avg_20d       | Bloomberg-style market data                  |
| Bond            | dur_adj_mid, convexity, yld_ytm_mid, rtg_sp | Bloomberg-style market data                  |
| FX              | px_last                                     | Bloomberg-style market data                  |
| Derivative      | delta, convexity, underlying price          | Bloomberg-style market data where applicable |


In [ ]:
position_based = pd.read_sql(
    text("""
    SELECT fund_id FROM funds 
    WHERE fund_id NOT IN ('AIFM_PE_Buyout', 'AIFM_Infra_Core')
    ORDER BY fund_id
    """),
    engine
)
position_based_funds = position_based['fund_id'].tolist()
position_based_funds


In [ ]:
from src.data.enrichment import enrich_positions

for fund in position_based_funds:
    # This calls MockBloomberg for each position with a ticker
    x = enrich_positions(engine, fund_id=fund, date='2026-05-13') 
    print()

## 4. Analytics input layer

`get_risk_ready_df()` loads the enriched position data into a pandas DataFrame.

```
Computation modules (src/computation/):
  ├─ var.py:          var_historical(positions_df, ...)
  ├─ stress.py:       stress_equity(positions_df, ...)
  ├─ liquidity.py:    liquidity_buckets(positions_df, ...)
  ├─ leverage.py:     compute_leverage(positions_df, ...)
  └─ attribution.py:  compute_pnl_attribution(positions_df, ...)
      ↓
Risk metrics (VaR, ES, stress outcomes, liquidity, leverage)
      ↓
Reporting (board_report, annex_iv) / Notebooks
```


In [ ]:
from src.data.enrichment import get_risk_ready_df

# Get enriched positions for a fund on a date
# This joins raw positions with enriched sensitivities
risk_df = get_risk_ready_df(engine, 'AIFM_HedgeFund', '2026-05-13')

print(f'Enriched positions: {len(risk_df)} rows')
print()
print('Key columns for risk computation:')
key_cols = ['isin', 'instrument_name', 'asset_class', 'market_value_eur', 'beta', 'dur_adj_mid', 'currency']
risk_df[key_cols].head()
